# CONSIST: Consistency Disparity Score Pipeline

Self-consistency disparity as an unsupervised metric for intersectional bias in LLMs.

**Runtime:** ~1-2 hours on free T4 (Colab) or P100 (Kaggle)

In [ ]:
# @title 1. Install Dependencies
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers datasets peft scipy matplotlib

import os, sys, json, random, warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings('ignore')

In [ ]:
# @title 2. Clone Package from GitHub
import os, sys, subprocess, shutil
REPO = "https://github.com/ArafathUIU/Research-Bias.git"
DST = "/content/consist"

if os.path.exists(DST):
    shutil.rmtree(DST, ignore_errors=True)
    print("Removed old cached copy")
subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/tmp_repo"], check=True)
shutil.move("/content/tmp_repo/consist", DST)
shutil.rmtree("/content/tmp_repo", ignore_errors=True)
print("Cloned fresh consist package from GitHub")

try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", "/content/consist", "--quiet"],
        check=True, capture_output=True,
    )
    print("Package installed via pip")
except Exception:
    sys.path.insert(0, "/content")
    print("Package added via sys.path (fallback)")

from consist.config import CONSISTConfig, INTERSECTIONAL_GROUPS
from consist.prompts import PromptGenerator, BIAS_DOMAINS, TEMPLATES
from consist.generate import GenerationHarness
from consist.embed import EmbeddingExtractor
from consist.cds import CDSCalculator
from consist.stats import StatisticalAnalyzer
from consist.validate import ValidationSuite
from consist.finetune import FineTuningIntervention
from consist.pipeline import CONSISTPipeline

In [ ]:
# @title 3. Configuration
cfg = CONSISTConfig(
    num_samples=20,
    temperatures=[0.3, 0.7, 1.0],
    n_bootstrap=1000,
    distance_metric="cosine",
    model_name="microsoft/Phi-3-mini-4k-instruct",  # swap to Llama/Gemma/Mistral with HF login
    embedding_model="sentence-transformers/all-MiniLM-L6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu",
    output_dir="results",
    seed=42,
)
print(f"Device: {cfg.device}")
print(f"Model: {cfg.model_name}")
print(f"Temperatures: {cfg.temperatures}")
print(f"Samples per prompt: {cfg.num_samples}")

In [ ]:
# @title 4. Initialize Pipeline
pipeline = CONSISTPipeline(cfg)
pipeline.setup()
print(f"Groups: {list(INTERSECTIONAL_GROUPS.keys())}")
print(f"Domains: {BIAS_DOMAINS}")

In [ ]:
# @title 5. Load Model
# Phi-3-mini is fully open (MIT) and needs no login.
# For gated models (Llama, Gemma, Mistral), first:
#   !huggingface-cli login --token YOUR_HF_TOKEN

pipeline.run_generation(model_name=cfg.model_name)
print("Model loaded successfully")

In [ ]:
# @title 6. Compute CDS at All Temperatures
results = pipeline.compute_all_temperatures()

for temp, result in results.items():
    print(f"\n=== T={temp} ===")
    print(f"  Overall CDS: {result.overall_cds():.4f}")
    for pair_key, val in result.aggregate_by_group_pair().items():
        print(f"  {pair_key}: {val:.4f}")

In [ ]:
# @title 7. Visualise Results
analyzer = pipeline.analyze(temperature=0.7)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

domain_summary = analyzer.domain_summary()
domains = list(domain_summary.keys())
d_means = [domain_summary[d]["mean"] for d in domains]
d_stds = [domain_summary[d]["std"] for d in domains]
axes[0].barh(domains, d_means, xerr=d_stds, capsize=4)
axes[0].axvline(0, color="gray", linestyle="--")
axes[0].set_xlabel("Mean CDS")
axes[0].set_title("CDS by Domain")

pair_summary = analyzer.group_pair_summary()
labels = list(pair_summary.keys())
p_means = [pair_summary[l]["mean"] for l in labels]
p_stds = [pair_summary[l]["std"] for l in labels]
axes[1].barh(labels, p_means, xerr=p_stds, capsize=4)
axes[1].axvline(0, color="gray", linestyle="--")
axes[1].set_xlabel("Mean CDS")
axes[1].set_title("CDS by Group Pair")

plt.tight_layout()
plt.savefig("cds_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# @title 8. Temperature Comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(pipeline.results[cfg.temperatures[0]].per_pair))
width = 0.25
colors = ["#4C72B0", "#DD8452", "#55A868"]

for i, (temp, res) in enumerate(sorted(pipeline.results.items())):
    vals = [p.cds_value for p in res.per_pair]
    ax.bar(x + i * width, vals, width, label=f"T={temp}", color=colors[i])

ax.set_xticks(x + width)
pair_labels = [f"{p.group_a[:4]}vs{p.group_b[:4]}" for p in pipeline.results[cfg.temperatures[0]].per_pair]
ax.set_xticklabels(pair_labels, rotation=45, ha="right")
ax.axhline(0, color="gray", linestyle="--")
ax.set_ylabel("CDS")
ax.set_title("CDS Across Temperatures")
ax.legend()
plt.tight_layout()
plt.savefig("cds_temperature_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# @title 9. Validation
# You'll need to fill in BBQ scores for your groups
# Example placeholders — replace with real data from your benchmark runs

BBQ_SCORES = {
    "Black_Female_vs_White_Male": 0.62,
    "Black_Male_vs_White_Male": 0.58,
    "Asian_Female_vs_White_Female": 0.65,
    "Hispanic_Male_vs_White_Male": 0.55,
}

DOWNSTREAM_SCORES = {
    "Black_Female_vs_White_Male": 0.71,
    "Black_Male_vs_White_Male": 0.67,
    "Asian_Female_vs_White_Female": 0.73,
    "Hispanic_Male_vs_White_Male": 0.64,
}

report = pipeline.validate(
    bbq_scores=BBQ_SCORES,
    downstream_scores=DOWNSTREAM_SCORES,
)

print("=== Validation Report ===")
print(f"CDS-BBQ correlation: {report.correlation_results.get('bbq_vs_cds', 'N/A')}")
print(f"Cross-model agreement: {report.cross_model_agreement or 'N/A'}")
if report.incremental_validity:
    iv = report.incremental_validity
    print(f"Incremental validity (r_bbq_only): {iv.get('r_bbq_only', 'N/A')}")
    print(f"Incremental validity (r_cds): {iv.get('r_cds_incremental', 'N/A')}")
    print(f"Incremental validity (p): {iv.get('p_cds_incremental', 'N/A')}")

In [ ]:
# @title 10. Fine-Tuning Intervention (Optional, ~20 min)
# Tests whether balanced fine-tuning reduces CDS disparity

RUN_FINETUNE = False  # set to True when ready

if RUN_FINETUNE:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        ),
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    tokenizer.pad_token = tokenizer.eos_token

    intervention = FineTuningIntervention(cfg)
    interven_result = intervention.run_intervention(
        model=model,
        tokenizer=tokenizer,
        cds_calculator=pipeline.cds_calculator,
        prompt_set=pipeline.prompt_set,
        sample_fn=lambda p: pipeline.generation_harness.generate_samples(
            p, num_samples=cfg.num_samples, temperature=0.7
        ),
        embed_fn=pipeline._embed_fn,
        n_per_group=100,
        num_epochs=3,
    )

    print(f"CDS before: {interven_result['cds_before']:.4f}")
    print(f"CDS after:  {interven_result['cds_after']:.4f}")
    print(f"Change:     {interven_result['cds_change']:.4f}")
    print(f"Training examples: {interven_result['n_train']}")
else:
    print("Skipping fine-tuning. Set RUN_FINETUNE = True to run.")

In [ ]:
# @title 11. Save & Export
pipeline.save_results()
!cp -r results /content/drive/MyDrive/Research_Bias/results_colab/
print("Results copied to Drive.")